In [ ]:
## ==================================================================================
## AFL SEASON SCRAPER - single-cell Colab version
## Scrapes player box-score stats + play-by-play for an entire AFL season and saves
## everything straight to Google Drive. Safe to re-run: already-scraped matches are
## skipped unless OVERWRITE=True. Progress is written to matches_index.csv as it goes.
## ==================================================================================

# ---------------- Config: edit these, then Runtime > Run all ----------------
SEASON        = 2026
DRIVE_FOLDER  = "AFL_Scraper"                 # folder created under My Drive
OUT_DIR       = f"/content/drive/MyDrive/{DRIVE_FOLDER}/{SEASON}"
HEADLESS      = True
DELAY_SECONDS = 2.0                            # politeness delay between matches
OVERWRITE     = False                          # re-scrape matches that already have output
MATCH_IDS     = None                           # e.g. "7150,7151" to bypass fixture discovery
MATCH_IDS_FILE = None                          # or a Drive path to a file, one match id per line
ONLY_MATCH_ID = None                           # set to a single match id (e.g. "7150") to test one match only

# ---------------- Install deps ----------------
import sys, subprocess
def _sh(*args):
    try:
        subprocess.run(args, check=True)
    except Exception as e:
        print("Command failed (continuing):", " ".join(args), "\n", e)

_sh(sys.executable, "-m", "pip", "install", "-q", "playwright", "requests")
_sh(sys.executable, "-m", "playwright", "install", "--with-deps", "chromium")

# ---------------- Mount Google Drive ----------------
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- Imports ----------------
import asyncio, csv, glob, hashlib, json, logging, re, time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple, TypeVar

import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("afl_scraper")

DEFAULT_USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124 Safari/537.36"
)

# =========================== common helpers ===========================

T = TypeVar("T")

def retry(fn, attempts=3, delay=2.0, backoff=2.0, on_error=None):
    last_exc = None
    wait = delay
    for i in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            last_exc = e
            if on_error:
                on_error(e, i)
            if i < attempts:
                time.sleep(wait)
                wait *= backoff
    raise last_exc

def deep_find_keys(node, targets, found):
    if not (targets - set(found.keys())):
        return
    if isinstance(node, dict):
        for k, v in node.items():
            if k in targets and k not in found:
                found[k] = v
            deep_find_keys(v, targets, found)
    elif isinstance(node, list):
        for v in node:
            deep_find_keys(v, targets, found)

def find_string_matching(node, pattern):
    if isinstance(node, str):
        return node if pattern.match(node) else None
    if isinstance(node, dict):
        for v in node.values():
            hit = find_string_matching(v, pattern)
            if hit:
                return hit
    elif isinstance(node, list):
        for v in node:
            hit = find_string_matching(v, pattern)
            if hit:
                return hit
    return None

CD_MATCH_CODE_RE = re.compile(r"^CD_M\d+$")

def find_cd_match_code(blobs):
    """Find the Champion Data match code (e.g. CD_M20260142207) embedded somewhere
    in the match page's captured JSON; that's what the matchPlays endpoint expects."""
    for blob in blobs:
        hit = find_string_matching(blob, CD_MATCH_CODE_RE)
        if hit:
            return hit
    return None

def get_in(obj, path, default=None):
    cur = obj
    for k in path:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def flatten_dict(d, parent="", sep="."):
    out = {}
    for k, v in d.items():
        nk = f"{parent}{sep}{k}" if parent else k
        if isinstance(v, dict):
            out.update(flatten_dict(v, nk, sep))
        elif isinstance(v, list):
            if v and all(isinstance(x, dict) for x in v):
                out[nk] = json.dumps(v, ensure_ascii=False)
            else:
                out[nk] = "|".join(map(str, v))
        else:
            out[nk] = v
    return out

def fingerprint_event(e):
    if "id" in e and isinstance(e["id"], (int, str)):
        return f"id::{e['id']}"
    canon = json.dumps(e, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return "sha1::" + hashlib.sha1(canon.encode("utf-8")).hexdigest()

def parse_mmss(s):
    if not isinstance(s, str):
        return float("inf")
    m = re.match(r"^(\d{1,2}):(\d{2})$", s.strip())
    if m:
        return int(m.group(1)) * 60 + int(m.group(2))
    try:
        return float(s)
    except Exception:
        return float("inf")

def write_csv(rows, path, base_cols=None):
    if not rows:
        open(path, "w", encoding="utf-8").close()
        return
    base_cols = base_cols or []
    dyn = []
    for r in rows:
        for k in r.keys():
            if k not in base_cols and k not in dyn:
                dyn.append(k)
    cols = base_cols + dyn
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in cols})

def write_ndjson(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# =========================== match player-stats scrape ===========================

PLAYER_STATS_BASE_COLS = [
    "name", "number", "position", "playerId", "teamId", "teamSide",
    "goals", "behinds", "kicks", "handballs", "disposals", "marks", "hitouts", "tackles",
    "centreClearances", "stoppageClearances", "totalClearances",
    "inside50s", "rebound50s",
    "freesFor", "freesAgainst",
    "contestedPossessions", "uncontestedPossessions",
    "marksInside50", "contestedMarks", "onePercenters", "bounces", "intercepts", "turnovers",
    "scoreInvolvements", "goalAssists", "shotsAtGoal",
    "disposalEfficiency", "metresGained", "dreamTeamPoints", "ratingPoints",
    "timeOnGroundPercentage", "rowLastUpdated",
    "effectiveKicks", "kickEfficiency", "kickToHandballRatio", "effectiveDisposals",
]

def match_url(match_id, base_url="https://www.afl.com.au/afl/matches"):
    return f"{base_url.rstrip('/')}/{match_id}"

def full_name(entry):
    given = get_in(entry, ["playerStats", "player", "playerName", "givenName"])
    sur = get_in(entry, ["playerStats", "player", "playerName", "surname"])
    if given or sur:
        return " ".join([x for x in [given, sur] if x]).strip()
    given = get_in(entry, ["player", "player", "playerName", "givenName"])
    sur = get_in(entry, ["player", "player", "playerName", "surname"])
    if given or sur:
        return " ".join([x for x in [given, sur] if x]).strip()
    nm = get_in(entry, ["playerStats", "player", "playerName"]) or get_in(entry, ["player", "player", "playerName"])
    return str(nm).strip() if nm else None

def flatten_stats_row(entry, side_label):
    stats = get_in(entry, ["playerStats", "stats"], {}) or {}
    ext = stats.get("extendedStats") or {}
    cl = stats.get("clearances") or {}
    row = {
        "name": full_name(entry),
        "number": get_in(entry, ["player", "jumperNumber"]),
        "position": get_in(entry, ["player", "player", "position"]),
        "playerId": get_in(entry, ["playerStats", "player", "playerId"]) or get_in(entry, ["player", "player", "playerId"]),
        "teamId": entry.get("teamId") or get_in(entry, ["playerStats", "teamId"]),
        "teamSide": side_label,
        "goals": stats.get("goals"), "behinds": stats.get("behinds"),
        "kicks": stats.get("kicks"), "handballs": stats.get("handballs"),
        "disposals": stats.get("disposals"), "marks": stats.get("marks"),
        "hitouts": stats.get("hitouts"), "tackles": stats.get("tackles"),
        "centreClearances": cl.get("centreClearances"),
        "stoppageClearances": cl.get("stoppageClearances"),
        "totalClearances": cl.get("totalClearances"),
        "inside50s": stats.get("inside50s"), "rebound50s": stats.get("rebound50s"),
        "freesFor": stats.get("freesFor"), "freesAgainst": stats.get("freesAgainst"),
        "contestedPossessions": stats.get("contestedPossessions"),
        "uncontestedPossessions": stats.get("uncontestedPossessions"),
        "metresGained": stats.get("metresGained") if "metresGained" in stats else stats.get("metersGained"),
        "marksInside50": stats.get("marksInside50"), "contestedMarks": stats.get("contestedMarks"),
        "onePercenters": stats.get("onePercenters"), "bounces": stats.get("bounces"),
        "intercepts": stats.get("intercepts"), "turnovers": stats.get("turnovers"),
        "scoreInvolvements": stats.get("scoreInvolvements"), "goalAssists": stats.get("goalAssists"),
        "shotsAtGoal": stats.get("shotsAtGoal"),
        "disposalEfficiency": stats.get("disposalEfficiency"),
        "dreamTeamPoints": stats.get("dreamTeamPoints"), "ratingPoints": stats.get("ratingPoints"),
        "timeOnGroundPercentage": get_in(entry, ["playerStats", "timeOnGroundPercentage"]),
        "rowLastUpdated": get_in(entry, ["playerStats", "lastUpdated"]),
    }
    for k in ("effectiveKicks", "kickEfficiency", "kickToHandballRatio", "effectiveDisposals"):
        if k in ext:
            row[k] = ext[k]
    return row

async def _capture_match_json(url, headless=True, settle_seconds=5.0):
    from playwright.async_api import async_playwright
    captured, pending = [], []
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=headless)
        context = await browser.new_context(user_agent=DEFAULT_USER_AGENT, locale="en-AU")
        page = await context.new_page()

        async def grab(resp):
            try:
                ct = (resp.headers or {}).get("content-type", "")
                rtype = resp.request.resource_type
                if ("application/json" in ct.lower()) or (rtype in ("xhr", "fetch")):
                    txt = await resp.text()
                    if txt:
                        try:
                            captured.append(json.loads(txt))
                        except Exception:
                            pass
            except Exception:
                pass

        page.on("response", lambda resp: pending.append(asyncio.create_task(grab(resp))))
        await page.goto(url, wait_until="domcontentloaded", timeout=60_000)
        try:
            await page.wait_for_load_state("networkidle", timeout=30_000)
        except Exception:
            pass
        await asyncio.sleep(settle_seconds)
        if pending:
            await asyncio.gather(*pending, return_exceptions=True)
        await browser.close()
    return captured

async def scrape_match_stats(match_id, out_dir=".", headless=True):
    match_id = str(match_id)
    url = match_url(match_id)
    out_path = Path(out_dir); out_path.mkdir(parents=True, exist_ok=True)

    captured = await _capture_match_json(url, headless=headless)

    targets = {"homeTeamPlayerStats", "awayTeamPlayerStats"}
    found = {}
    for blob in captured:
        deep_find_keys(blob, targets, found)
        if targets.issubset(found.keys()):
            break

    home = found.get("homeTeamPlayerStats") or []
    away = found.get("awayTeamPlayerStats") or []
    home_rows = [flatten_stats_row(x, "home") for x in home]
    away_rows = [flatten_stats_row(x, "away") for x in away]
    all_rows = home_rows + away_rows

    home_csv = str(out_path / f"match{match_id}_home_player_stats.csv")
    away_csv = str(out_path / f"match{match_id}_away_player_stats.csv")
    all_csv  = str(out_path / f"match{match_id}_all_player_stats.csv")
    write_csv(home_rows, home_csv, PLAYER_STATS_BASE_COLS)
    write_csv(away_rows, away_csv, PLAYER_STATS_BASE_COLS)
    write_csv(all_rows, all_csv, PLAYER_STATS_BASE_COLS)

    cd_code = find_cd_match_code(captured)
    return {
        "match_id": match_id, "home_rows": home_rows, "away_rows": away_rows,
        "cd_match_code": cd_code,
        "csv_paths": {"home": home_csv, "away": away_csv, "all": all_csv},
    }

# =========================== play-by-play scrape ===========================

TOKEN_URL = "https://api.afl.com.au/cfs/afl/WMCTok"
MATCH_PLAYS_URL = "https://sapi.afl.com.au/afl/matchPlays/{code}"

class TokenExpiredError(RuntimeError):
    pass

def get_afl_token(timeout=15.0):
    def _fetch():
        r = requests.post(TOKEN_URL, timeout=timeout)
        r.raise_for_status()
        tok = r.json().get("token")
        if not tok:
            raise RuntimeError("No 'token' in WMCTok response")
        return tok
    return retry(_fetch, attempts=3, delay=2.0)

def fetch_json_with_token(url, token, timeout=30.0):
    headers = {
        "Accept": "application/json, text/plain, */*",
        "User-Agent": DEFAULT_USER_AGENT,
        "x-media-mis-token": token,
        "Origin": "https://www.afl.com.au",
        "Referer": "https://www.afl.com.au/",
    }
    r = requests.get(url, headers=headers, timeout=timeout)
    if r.status_code == 401:
        raise TokenExpiredError(f"401 Unauthorized fetching {url}")
    r.raise_for_status()
    return r.json()

def fetch_match_plays_json(code, token=None):
    token = token or get_afl_token()
    url = MATCH_PLAYS_URL.format(code=code)
    try:
        data = retry(lambda: fetch_json_with_token(url, token), attempts=2, delay=2.0)
    except TokenExpiredError:
        token = get_afl_token()
        data = retry(lambda: fetch_json_with_token(url, token), attempts=2, delay=2.0)
    return data, token

def _iter_nodes_with_path(root, path="$"):
    yield path, root
    if isinstance(root, dict):
        for k, v in root.items():
            if isinstance(v, (dict, list)):
                yield from _iter_nodes_with_path(v, f"{path}.{k}")
    elif isinstance(root, list):
        for i, v in enumerate(root):
            if isinstance(v, (dict, list)):
                yield from _iter_nodes_with_path(v, f"{path}[{i}]")

def _eventish_score(d):
    keys = " ".join(map(str, d.keys())).lower()
    return sum(1 for hint in ("period", "quarter", "qtr", "time", "display", "type", "event", "team", "player", "x", "y", "score") if hint in keys)

def collect_event_lists(root):
    found = []
    for p, node in _iter_nodes_with_path(root):
        if isinstance(node, list) and node and isinstance(node[0], dict):
            sample = node[:5]
            avg = sum(_eventish_score(x) for x in sample) / len(sample)
            if avg >= 2:
                found.append((p, node))
    priority = (".$.plays", ".$.matchPlays", ".$.events", ".$.data")
    found.sort(key=lambda item: 0 if any(item[0].endswith(k[1:]) for k in priority) else 1)
    return found

def row_period(flat):
    for k in ("period.number", "quarter", "qtr", "period"):
        v = flat.get(k)
        if isinstance(v, int):
            return v
        if isinstance(v, str) and v.isdigit():
            return int(v)
    return 0

def row_seconds(flat):
    for k in ("period.secondsRemaining", "secondsRemaining", "seconds", "timeSeconds"):
        v = flat.get(k)
        if isinstance(v, (int, float)):
            return float(v)
        if isinstance(v, str) and v.isdigit():
            return float(v)
    for k in ("displayTime", "period.displayTime", "time", "clock"):
        v = flat.get(k)
        if v is not None:
            return parse_mmss(str(v))
    return float("inf")

def scrape_match_plays(code, out_dir=".", token=None):
    out_path = Path(out_dir); out_path.mkdir(parents=True, exist_ok=True)
    data, _token = fetch_match_plays_json(code, token=token)

    raw_path = str(out_path / f"{code}_raw.json")
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    lists = collect_event_lists(data)

    primary_csv_path = None
    if lists:
        primary_path, primary_list = lists[0]
        primary_flat = []
        for ev in primary_list:
            flat = flatten_dict(ev); flat["_sourcePath"] = primary_path
            primary_flat.append(flat)
        primary_csv_path = str(out_path / f"{code}_plays.csv")
        write_csv(primary_flat, primary_csv_path)

    seen, all_flat = set(), []
    for p, arr in lists:
        for ev in arr:
            fp = fingerprint_event(ev)
            if fp in seen:
                continue
            seen.add(fp)
            flat = flatten_dict(ev); flat["_sourcePath"] = p
            all_flat.append(flat)
    all_flat.sort(key=lambda r: (row_period(r), row_seconds(r)))

    all_csv = str(out_path / f"{code}_plays_all.csv")
    all_json = str(out_path / f"{code}_plays_all.ndjson")
    write_csv(all_flat, all_csv)
    write_ndjson([e for _, arr in lists for e in arr], all_json)

    csv_paths = {"raw": raw_path, "all": all_csv, "ndjson": all_json}
    if primary_csv_path:
        csv_paths["primary"] = primary_csv_path
    return {"cd_match_code": code, "n_events": len(all_flat), "csv_paths": csv_paths}

# =========================== season fixture discovery ===========================

MATCH_LINK_RE = re.compile(r"/afl/matches/(\d+)")

@dataclass
class MatchRef:
    match_id: str
    round: Optional[str] = None
    home_team: Optional[str] = None
    away_team: Optional[str] = None
    utc_start_time: Optional[str] = None

    def year(self):
        if not self.utc_start_time:
            return None
        m = re.match(r"^(\d{4})-", self.utc_start_time)
        return int(m.group(1)) if m else None

def _match_list_score(d):
    keys = " ".join(map(str, d.keys())).lower()
    return sum(1 for hint in ("round", "home", "away", "venue", "date", "starttime", "utc", "match", "team") if hint in keys)

def _iter_nodes(root):
    yield root
    if isinstance(root, dict):
        for v in root.values():
            if isinstance(v, (dict, list)):
                yield from _iter_nodes(v)
    elif isinstance(root, list):
        for v in root:
            if isinstance(v, (dict, list)):
                yield from _iter_nodes(v)

def _collect_match_lists(root):
    found = []
    for node in _iter_nodes(root):
        if isinstance(node, list) and node and isinstance(node[0], dict):
            sample = node[:5]
            avg = sum(_match_list_score(x) for x in sample) / len(sample)
            if avg >= 2:
                found.append(node)
    return found

def _extract_match_ref(entry):
    match_id = entry.get("matchId") or entry.get("id") or get_in(entry, ["match", "id"]) or get_in(entry, ["match", "matchId"])
    if match_id is None or not str(match_id).isdigit():
        return None
    home = get_in(entry, ["homeTeam", "name", "abbreviation"]) or get_in(entry, ["homeTeam", "abbreviation"]) or get_in(entry, ["homeTeam", "name"])
    away = get_in(entry, ["awayTeam", "name", "abbreviation"]) or get_in(entry, ["awayTeam", "abbreviation"]) or get_in(entry, ["awayTeam", "name"])
    round_name = entry.get("roundName") or entry.get("round") or get_in(entry, ["round", "name"])
    start = entry.get("utcStartTime") or entry.get("date") or entry.get("startDateTime")
    return MatchRef(
        match_id=str(match_id),
        round=str(round_name) if round_name is not None else None,
        home_team=str(home) if home else None,
        away_team=str(away) if away else None,
        utc_start_time=str(start) if start else None,
    )

async def _load_page_json_and_links(url, headless=True, settle_seconds=5.0):
    from playwright.async_api import async_playwright
    captured, pending = [], []
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=headless)
        context = await browser.new_context(user_agent=DEFAULT_USER_AGENT, locale="en-AU")
        page = await context.new_page()

        async def grab(resp):
            try:
                ct = (resp.headers or {}).get("content-type", "")
                rtype = resp.request.resource_type
                if ("application/json" in ct.lower()) or (rtype in ("xhr", "fetch")):
                    txt = await resp.text()
                    if txt:
                        try:
                            captured.append(json.loads(txt))
                        except Exception:
                            pass
            except Exception:
                pass

        page.on("response", lambda resp: pending.append(asyncio.create_task(grab(resp))))
        await page.goto(url, wait_until="domcontentloaded", timeout=60_000)
        try:
            await page.wait_for_load_state("networkidle", timeout=30_000)
        except Exception:
            pass
        await asyncio.sleep(settle_seconds)
        if pending:
            await asyncio.gather(*pending, return_exceptions=True)
        hrefs = await page.eval_on_selector_all("a[href*='/afl/matches/']", "els => els.map(e => e.getAttribute('href'))")
        await browser.close()
    return captured, hrefs

def candidate_fixture_urls(season):
    return [
        f"https://www.afl.com.au/fixture?Season={season}",
        f"https://www.afl.com.au/fixture/{season}",
        "https://www.afl.com.au/fixture",
    ]

async def discover_season_matches(season, headless=True, extra_urls=None):
    urls = (extra_urls or []) + candidate_fixture_urls(season)
    by_id, link_only_ids = {}, set()
    for url in urls:
        try:
            blobs, hrefs = await _load_page_json_and_links(url, headless=headless)
        except Exception:
            continue
        for blob in blobs:
            for lst in _collect_match_lists(blob):
                for entry in lst:
                    ref = _extract_match_ref(entry)
                    if ref is not None and ref.match_id not in by_id:
                        by_id[ref.match_id] = ref
        for href in hrefs:
            m = MATCH_LINK_RE.search(href or "")
            if m:
                link_only_ids.add(m.group(1))
        if by_id or link_only_ids:
            break

    matches = [ref for ref in by_id.values() if ref.year() in (None, season)]
    known_ids = {m.match_id for m in matches}
    for mid in link_only_ids - known_ids:
        matches.append(MatchRef(match_id=mid))
    matches.sort(key=lambda m: (m.round or "", m.match_id))
    return matches

def load_match_ids_file(path):
    refs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split(",")]
            refs.append(MatchRef(
                match_id=parts[0],
                round=parts[1] if len(parts) > 1 else None,
                home_team=parts[2] if len(parts) > 2 else None,
                away_team=parts[3] if len(parts) > 3 else None,
            ))
    return refs

# =========================== season orchestration ===========================

@dataclass
class MatchOutcome:
    match_id: str
    round: Optional[str] = None
    home_team: Optional[str] = None
    away_team: Optional[str] = None
    stats_ok: bool = False
    plays_ok: bool = False
    cd_match_code: Optional[str] = None
    error: str = ""

def _match_dir(out_dir, match_id):
    return out_dir / str(match_id)

def _already_scraped(match_dir, match_id):
    stats_done = (match_dir / f"match{match_id}_all_player_stats.csv").exists()
    plays_done = any(match_dir.glob("CD_M*_plays_all.csv"))
    return stats_done and plays_done

async def scrape_one_match(ref, out_dir, headless=True, overwrite=False, token=None):
    outcome = MatchOutcome(match_id=ref.match_id, round=ref.round, home_team=ref.home_team, away_team=ref.away_team)
    match_dir = _match_dir(out_dir, ref.match_id)

    if not overwrite and _already_scraped(match_dir, ref.match_id):
        outcome.stats_ok = outcome.plays_ok = True
        logger.info("match %s already scraped, skipping", ref.match_id)
        return outcome

    try:
        stats_result = await scrape_match_stats(ref.match_id, out_dir=str(match_dir), headless=headless)
        outcome.stats_ok = True
        outcome.cd_match_code = stats_result["cd_match_code"]
    except Exception as e:
        outcome.error = f"stats: {e}"
        logger.warning("match %s stats scrape failed: %s", ref.match_id, e)
        return outcome

    if not outcome.cd_match_code:
        outcome.error = "could not find CD_M... match code on the stats page; skipped plays scrape"
        logger.warning("match %s: %s", ref.match_id, outcome.error)
        return outcome

    try:
        plays_result = await asyncio.to_thread(scrape_match_plays, outcome.cd_match_code, str(match_dir), token)
        outcome.plays_ok = True
    except Exception as e:
        outcome.error = f"plays: {e}"
        logger.warning("match %s plays scrape failed: %s", ref.match_id, e)

    return outcome

def _write_matches_index(out_dir, outcomes):
    path = out_dir / "matches_index.csv"
    cols = ["match_id", "round", "home_team", "away_team", "stats_ok", "plays_ok", "cd_match_code", "error"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for o in outcomes:
            w.writerow({
                "match_id": o.match_id, "round": o.round or "",
                "home_team": o.home_team or "", "away_team": o.away_team or "",
                "stats_ok": o.stats_ok, "plays_ok": o.plays_ok,
                "cd_match_code": o.cd_match_code or "", "error": o.error,
            })

def _concat_csvs(pattern, out_path):
    paths = sorted(glob.glob(pattern))
    n_rows, header, writer = 0, None, None
    with open(out_path, "w", newline="", encoding="utf-8") as out_f:
        for p in paths:
            if not Path(p).stat().st_size:
                continue
            with open(p, newline="", encoding="utf-8") as in_f:
                rows = list(csv.reader(in_f))
            if not rows:
                continue
            if header is None:
                header = rows[0]
                writer = csv.writer(out_f)
                writer.writerow(header)
            for row in rows[1:]:
                writer.writerow(row)
                n_rows += 1
    return n_rows

async def scrape_season(season, out_dir, headless=True, overwrite=False, delay_seconds=2.0,
                         match_ids_file=None, matches=None):
    out_path = Path(out_dir); out_path.mkdir(parents=True, exist_ok=True)

    if matches is not None:
        refs = matches
    elif match_ids_file:
        refs = load_match_ids_file(match_ids_file)
    else:
        refs = await discover_season_matches(season, headless=headless)

    if not refs:
        raise RuntimeError(
            f"No matches discovered for season {season}. Fixture-page heuristics may need "
            "adjusting for the current site -- set MATCH_IDS or MATCH_IDS_FILE above to bypass "
            "discovery with a manual list."
        )

    logger.info("season %s: %d matches to scrape -> %s", season, len(refs), out_path)

    outcomes = []
    token = None
    for i, ref in enumerate(refs):
        outcome = await scrape_one_match(ref, out_path, headless=headless, overwrite=overwrite, token=token)
        outcomes.append(outcome)
        _write_matches_index(out_path, outcomes)
        if i < len(refs) - 1:
            time.sleep(delay_seconds)

    n_stats_ok = sum(o.stats_ok for o in outcomes)
    n_plays_ok = sum(o.plays_ok for o in outcomes)
    logger.info("season %s done: %d/%d stats ok, %d/%d plays ok", season, n_stats_ok, len(outcomes), n_plays_ok, len(outcomes))

    n_players = _concat_csvs(str(out_path / "*" / "match*_all_player_stats.csv"), out_path / f"season_{season}_player_stats.csv")
    n_plays = _concat_csvs(str(out_path / "*" / "CD_M*_plays_all.csv"), out_path / f"season_{season}_plays.csv")
    logger.info("season %s: wrote %d player-stat rows and %d play rows to season-level CSVs", season, n_players, n_plays)

    return outcomes

# =========================== run ===========================

if ONLY_MATCH_ID:
    print(f"Scraping single match {ONLY_MATCH_ID} into {OUT_DIR} ...")
    _stats = await scrape_match_stats(ONLY_MATCH_ID, out_dir=OUT_DIR, headless=HEADLESS)
    print(f"Player stats: home={len(_stats['home_rows'])} away={len(_stats['away_rows'])} -> {_stats['csv_paths']}")
    if _stats["cd_match_code"]:
        _plays = scrape_match_plays(_stats["cd_match_code"], out_dir=OUT_DIR)
        print(f"Play-by-play: {_plays['n_events']} events -> {_plays['csv_paths']}")
    else:
        print("No CD_M... code found on the page; play-by-play scrape skipped.")
else:
    _matches = None
    if MATCH_IDS:
        _matches = [MatchRef(match_id=m.strip()) for m in MATCH_IDS.split(",") if m.strip()]

    print(f"Scraping full {SEASON} season into {OUT_DIR} ...")
    _outcomes = await scrape_season(
        SEASON, OUT_DIR, headless=HEADLESS, overwrite=OVERWRITE,
        delay_seconds=DELAY_SECONDS, match_ids_file=MATCH_IDS_FILE, matches=_matches,
    )
    _n_stats_ok = sum(o.stats_ok for o in _outcomes)
    _n_plays_ok = sum(o.plays_ok for o in _outcomes)
    print(f"\n{SEASON} season: {len(_outcomes)} matches discovered, {_n_stats_ok} stats scraped, {_n_plays_ok} plays scraped.")
    _failed = [o for o in _outcomes if not (o.stats_ok and o.plays_ok)]
    if _failed:
        print(f"{len(_failed)} match(es) had issues (see {OUT_DIR}/matches_index.csv):")
        for o in _failed[:20]:
            print(f"  - match {o.match_id}: {o.error}")
